# NB-7 — H-1: the freeze/intervene audit on standard Houlsby adapters (DeBERTa-v3-large)

Implements `FMI_manifest_H1_standard_adapters.md`. Houlsby bottleneck adapters (bottleneck 10, 48 adapters, 1,032,672
parameters) are trained (a) on a frozen backbone and (b) with matched full fine-tuning; every selected checkpoint is scored
once on the evaluation set with all adapters **on** and with all adapters **off** (exact bypass). The archived no-adapter
full-fine-tuning runs serve as the contribution baseline.

Set `TASK` in cell 3: run once with `TASK = 'boolq'` (3 seeds, ~5–6 h) and once with `TASK = 'winogrande'` (5 seeds,
~6–7 h). Run first with `SMOKE_TEST = True` (a few minutes) to check the pipeline. A100 + High-RAM. Restart-safe: completed
(regime, seed) runs are committed to Drive and skipped on re-run. The runtime is not deleted.

In [ ]:
import time
from datetime import datetime, timezone
RUN_STARTED_PERF = time.perf_counter(); RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
from google.colab import drive
drive.mount('/content/drive'); print('Run started (UTC):', RUN_STARTED_UTC)

In [ ]:
import os, subprocess, sys
# keep transformers away from the TensorFlow/Flax stacks preinstalled on Colab (their protobuf builds can conflict)
os.environ['USE_TF'] = '0'; os.environ['USE_FLAX'] = '0'; os.environ['USE_TORCH'] = '1'; os.environ['TRANSFORMERS_NO_TF'] = '1'
packages = ['pandas==2.2.3', 'transformers==4.57.6', 'datasets==3.2.0', 'huggingface-hub==0.36.2', 'accelerate>=1.10,<2',
            'sentencepiece>=0.2,<0.3', 'scikit-learn==1.6.1', 'requests']
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if proc.returncode: print(proc.stdout); raise RuntimeError('Dependency installation failed.')
print('Dependencies installed.')

In [ ]:
import os
os.environ['USE_TF'] = '0'; os.environ['USE_FLAX'] = '0'; os.environ['USE_TORCH'] = '1'; os.environ['TRANSFORMERS_NO_TF'] = '1'
import gc, hashlib, json, math, random, shutil, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
warnings.filterwarnings('ignore', category=FutureWarning); os.environ['TOKENIZERS_PARALLELISM'] = 'false'
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not torch.cuda.is_available(): raise RuntimeError('GPU required (A100 recommended).')
DEVICE = torch.device('cuda'); AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16; GPU_NAME = torch.cuda.get_device_name(0)
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True; torch.set_float32_matmul_precision('high')

TASK = 'boolq'                 # 'boolq' or 'winogrande'
SMOKE_TEST = True              # True: tiny subsample, 1 epoch, seed 13 only. False: full study.
SAVE_FULL_FT_STATES = True     # full fine-tuning states are ~1.7 GB each; set False if Drive space is short (adapters + scorer are always saved)

MANIFEST_ID = 'FMI_manifest_H1_standard_adapters.md'
assert TASK in ('boolq', 'winogrande')
SEEDS = [13, 42, 71] if TASK == 'boolq' else [13, 42, 71, 7595, 1676]
HEAD_ONLY_MEAN_BA = 0.6046690798271527 if TASK == 'boolq' else 0.7486
DATA_SEED, EPOCHS, MAX_LENGTH = 2026, 3, 512
BOTTLENECK = 10
BATCH_SIZE, EVAL_BATCH_SIZE, GRAD_ACCUM_STEPS = 2, 4, 8
LR_FROZEN = 2e-4; ENCODER_LR = 8e-6; ADAPTER_SCORER_LR = 2e-4; WEIGHT_DECAY = 0.01; WARMUP_RATIO = 0.06
BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES = 20261002, 10000
REGIMES = ['frozen', 'fullft']
if SMOKE_TEST: SEEDS = [13]; EPOCHS = 1

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large'); LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
DRIVE_ROOT = Path('/content/drive/MyDrive/FMI') / (f'NB7_houlsby_{TASK}' + ('_smoke' if SMOKE_TEST else '')); PROGRESS = DRIVE_ROOT / 'progress'; STATES = DRIVE_ROOT / 'states'
WORK = Path('/content/fmi_nb7'); RESULT_DIR = WORK / 'results'; DATA_DIR = WORK / 'data'; REPO_DIR = WORK / 'repo'
for p in (WORK, RESULT_DIR, DATA_DIR, PROGRESS, STATES): p.mkdir(parents=True, exist_ok=True)
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(64 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
required = ['config.json', 'spm.model', 'tokenizer_config.json']
if any(not (DRIVE_MODEL_DIR / n).is_file() for n in required) or not any((DRIVE_MODEL_DIR / n).is_file() for n in ('model.safetensors', 'pytorch_model.bin')):
    raise FileNotFoundError(f'Incomplete base model folder: {DRIVE_MODEL_DIR}')
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True); MODEL_NAME = str(LOCAL_MODEL_DIR)
# archived no-adapter full-fine-tuning predictions (contribution baseline) from the public repository
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kurbankotan/freeze-match-intervene-audit.git', str(REPO_DIR)], text=True, capture_output=True)
if r.returncode: print(r.stderr); raise RuntimeError('git clone failed')
REPO_COMMIT = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True, capture_output=True).stdout.strip()
print('GPU:', GPU_NAME, '| AMP', AMP_DTYPE, '| torch', torch.__version__, '| TASK', TASK, '| seeds', SEEDS, '| SMOKE_TEST', SMOKE_TEST, '| repo', REPO_COMMIT[:7])

In [ ]:
# Data: exactly the splits, serializations and orders of the paper. Each item: two candidates, base semantics, semantic gold, cluster id.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
def pair_items(first_a, second_a, first_b, second_b, golds, base_semantics):
    ea = tokenizer(first_a, second_a, truncation='only_second' if second_a is not None else True, max_length=MAX_LENGTH) if second_a is not None else tokenizer(first_a, truncation=True, max_length=MAX_LENGTH)
    eb = tokenizer(first_b, second_b, truncation='only_second' if second_b is not None else True, max_length=MAX_LENGTH) if second_b is not None else tokenizer(first_b, truncation=True, max_length=MAX_LENGTH)
    return [{'c': [{'input_ids': ea['input_ids'][i], 'attention_mask': ea['attention_mask'][i]}, {'input_ids': eb['input_ids'][i], 'attention_mask': eb['attention_mask'][i]}],
             'sem': list(base_semantics), 'gold': int(golds[i]), 'index': i} for i in range(len(golds))]
if TASK == 'boolq':
    seed_everything(DATA_SEED)
    raw = load_dataset('google/boolq', token=HF_TOKEN); train_full = raw['train']; heldout = raw['validation']
    groups = np.asarray(train_full['passage']); labels = np.asarray(train_full['answer'], dtype=int)
    splitter = GroupShuffleSplit(n_splits=80, test_size=1075 / len(train_full), random_state=DATA_SEED); best = None
    for tr, va in splitter.split(np.zeros(len(labels)), labels, groups):
        score = abs(len(va) - 1075) / 1075 + abs(labels[va].mean() - labels.mean())
        if best is None or score < best[0]: best = (score, tr, va)
    _, tr_idx, va_idx = best
    tr_rows = train_full.select(tr_idx.tolist()); va_rows = train_full.select(va_idx.tolist())
    assert len(tr_rows) == 8351 and len(va_rows) == 1076 and len(heldout) == 3270 and set(tr_rows['passage']).isdisjoint(set(va_rows['passage']))
    def build(rows):
        qs = [f'question: {q} answer:' for q in rows['question']]
        return pair_items([q + ' yes' for q in qs], rows['passage'], [q + ' no' for q in qs], rows['passage'], [int(bool(a)) for a in rows['answer']], [1, 0])
    if SMOKE_TEST: tr_rows, va_rows, heldout = tr_rows.select(range(160)), va_rows.select(range(64)), heldout.select(range(200))
    ENC = {'train': build(tr_rows), 'val': build(va_rows), 'eval': build(heldout)}
    hh = [hashlib.sha256(p.encode('utf-8')).hexdigest() for p in heldout['passage']]
else:
    zip_local = DATA_DIR / 'winogrande_1.1.zip'
    if not zip_local.is_file():
        resp = requests.get('https://storage.googleapis.com/ai2-mosaic/public/winogrande/winogrande_1.1.zip', timeout=120); resp.raise_for_status(); zip_local.write_bytes(resp.content)
    def read_split(zf, name):
        rows = [json.loads(l) for l in zf.read(f'winogrande_1.1/{name}.jsonl').decode('utf-8').splitlines() if l.strip()]
        labs = zf.read(f'winogrande_1.1/{name}-labels.lst').decode('utf-8').split()
        return [{'qID': r_['qID'], 'cluster': r_['qID'].rsplit('-', 1)[0], 'sentence': r_['sentence'], 'option1': r_['option1'], 'option2': r_['option2'], 'gold': int(l_) - 1} for r_, l_ in zip(rows, labs)]
    with zipfile.ZipFile(zip_local) as zf: train_all = read_split(zf, 'train_l'); heldout = read_split(zf, 'dev')
    assert len(train_all) == 10234 and len(heldout) == 1267
    seed_everything(DATA_SEED)
    groups = np.array([r_['cluster'] for r_ in train_all]); labels = np.array([r_['gold'] for r_ in train_all])
    splitter = GroupShuffleSplit(n_splits=80, test_size=0.10, random_state=DATA_SEED); best = None
    for tr, va in splitter.split(np.zeros(len(labels)), labels, groups):
        score = abs(len(va) / len(labels) - 0.10) + abs(labels[va].mean() - labels.mean())
        if best is None or score < best[0]: best = (score, tr, va)
    _, tr_idx, va_idx = best
    tr_rows = [train_all[i] for i in tr_idx]; va_rows = [train_all[i] for i in va_idx]
    if SMOKE_TEST: tr_rows, va_rows, heldout = tr_rows[:160], va_rows[:64], heldout[:200]
    def build(rows):
        fill = lambda r_, o: r_['sentence'].replace('_', o)
        return pair_items([fill(r_, r_['option1']) for r_ in rows], None, [fill(r_, r_['option2']) for r_ in rows], None, [r_['gold'] for r_ in rows], [0, 1])
    ENC = {'train': build(tr_rows), 'val': build(va_rows), 'eval': build(heldout)}
    hh = [r_['cluster'] for r_ in heldout]
_cl = {h: i for i, h in enumerate(sorted(set(hh)))}; CLUSTER = np.array([_cl[h] for h in hh]); N_CLUSTERS = len(_cl)
GOLD = np.array([it['gold'] for it in ENC['eval']])
tr_gold = np.array([it['gold'] for it in ENC['train']]); n1 = int(tr_gold.sum()); n0 = len(tr_gold) - n1
CLASS_WEIGHT = torch.tensor([len(tr_gold) / (2 * max(n0, 1)), len(tr_gold) / (2 * max(n1, 1))], dtype=torch.float32, device=DEVICE)

class PairDataset(Dataset):
    def __init__(self, items, training=False, seed=0): self.items, self.training, self.seed, self.epoch = items, training, int(seed), 0
    def set_epoch(self, e): self.epoch = int(e)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]; swap = False
        if self.training: swap = random.Random(self.seed * 1000003 + self.epoch * 10007 + i).random() < 0.5
        order = [1, 0] if swap else [0, 1]; sem = [it['sem'][k] for k in order]
        return {'candidates': [it['c'][k] for k in order], 'semantics': sem, 'label': sem.index(it['gold']), 'gold': it['gold'], 'index': it['index']}
def collate(batch):
    flat = [c for b in batch for c in b['candidates']]; padded = tokenizer.pad(flat, padding=True, return_tensors='pt'); bsz = len(batch)
    return {'input_ids': padded['input_ids'].view(bsz, 2, -1), 'attention_mask': padded['attention_mask'].view(bsz, 2, -1), 'semantics': torch.tensor([b['semantics'] for b in batch]),
            'labels': torch.tensor([b['label'] for b in batch]), 'gold': torch.tensor([b['gold'] for b in batch]), 'index': torch.tensor([b['index'] for b in batch])}
def loader(split, seed, training=False):
    ds = PairDataset(ENC[split], training=training, seed=seed); g = torch.Generator().manual_seed(seed)
    return ds, DataLoader(ds, batch_size=BATCH_SIZE if training else EVAL_BATCH_SIZE, shuffle=training, collate_fn=collate, num_workers=2, pin_memory=True, generator=g)
print({'task': TASK, 'train': len(ENC['train']), 'val': len(ENC['val']), 'eval': len(ENC['eval']), 'eval_clusters': int(N_CLUSTERS), 'class_weight': CLASS_WEIGHT.tolist()})

In [ ]:
# Houlsby adapters: after the output projection of every attention and feed-forward sublayer, before the residual sum and LayerNorm.
class HoulsbyAdapter(nn.Module):
    def __init__(self, hidden, bottleneck):
        super().__init__(); self.down = nn.Linear(hidden, bottleneck); self.up = nn.Linear(bottleneck, hidden); self.enabled = True
        nn.init.normal_(self.down.weight, std=1e-2); nn.init.zeros_(self.down.bias); nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, y):
        if not self.enabled: return y
        return y + self.up(F.gelu(self.down(y)))
class AdaptedDense(nn.Module):
    def __init__(self, dense, bottleneck):
        super().__init__(); self.dense = dense; self.adapter = HoulsbyAdapter(dense.out_features, bottleneck)
    def forward(self, x): return self.adapter(self.dense(x))
def inject_houlsby(encoder, bottleneck):
    adapters = []
    for layer in encoder.encoder.layer:
        layer.attention.output.dense = AdaptedDense(layer.attention.output.dense, bottleneck); adapters.append(layer.attention.output.dense.adapter)
        layer.output.dense = AdaptedDense(layer.output.dense, bottleneck); adapters.append(layer.output.dense.adapter)
    return adapters
class HoulsbyModel(nn.Module):
    def __init__(self, regime):
        super().__init__(); self.regime = regime
        self.encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        if hasattr(self.encoder.config, 'use_cache'): self.encoder.config.use_cache = False
        self.adapters = inject_houlsby(self.encoder, BOTTLENECK)
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        self.dropout = nn.Dropout(0.15); self.scorer = nn.Linear(int(self.encoder.config.hidden_size), 1)
        for n, p in self.encoder.named_parameters():
            p.requires_grad = (regime == 'fullft') or ('.adapter.' in n)
    def set_adapters(self, on):
        for a in self.adapters: a.enabled = bool(on)
    def adapter_params(self): return [p for a in self.adapters for p in a.parameters()]
    def encoder_core_params(self): return [p for n, p in self.encoder.named_parameters() if '.adapter.' not in n]
    def forward(self, ids, mask, state='on'):
        self.set_adapters(state == 'on'); h = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state; self.set_adapters(True)
        return self.scorer(self.dropout(h[:, 0]))
_probe = HoulsbyModel('frozen')
N_ADAPTER = sum(p.numel() for p in _probe.adapter_params()); N_TRAINABLE_FROZEN = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
assert len(_probe.adapters) == 48 and N_ADAPTER == 48 * (2 * 1024 * BOTTLENECK + BOTTLENECK + 1024) == 1032672, N_ADAPTER
assert N_TRAINABLE_FROZEN == N_ADAPTER + 1025, N_TRAINABLE_FROZEN
del _probe; gc.collect(); print('adapters: 48 | adapter parameters', N_ADAPTER, '| trainable in frozen regime', N_TRAINABLE_FROZEN)

In [ ]:
# Training and evaluation (locked settings); selection on internal validation with adapters on.
def metrics(gold, pred, prob, loss):
    return {'loss': float(loss), 'accuracy': accuracy_score(gold, pred), 'balanced_accuracy': balanced_accuracy_score(gold, pred), 'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
            'predicted_sem1_rate': float(np.mean(pred)), 'log_loss': float(log_loss(gold, np.clip(prob, 1e-7, 1 - 1e-7), labels=[0, 1]))}
@torch.no_grad()
def evaluate(model, loader_, states):
    model.eval(); tot = {st: {'loss': 0.0, 'gold': [], 'pred': [], 'prob1': [], 'index': []} for st in states}
    for batch in loader_:
        ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); sem = batch['semantics']; bsz, nopt, L = ids.shape
        for st in states:
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids.view(bsz * nopt, L), mask.view(bsz * nopt, L), state=st).view(bsz, nopt).float()
            t = tot[st]; t['loss'] += float(F.cross_entropy(logits, labels, reduction='sum')); probs = torch.softmax(logits, dim=-1).cpu(); pi = logits.argmax(dim=-1).cpu()
            t['pred'].extend(sem.gather(1, pi[:, None]).squeeze(1).numpy().tolist()); t['prob1'].extend(probs.gather(1, (sem == 1).long().argmax(dim=1, keepdim=True)).squeeze(1).numpy().tolist())
            t['gold'].extend(batch['gold'].numpy().tolist()); t['index'].extend(batch['index'].numpy().tolist())
    out = {}
    for st, v in tot.items():
        o = np.argsort(np.asarray(v['index'])); g, p, pr = np.asarray(v['gold'])[o], np.asarray(v['pred'])[o], np.asarray(v['prob1'])[o]
        out[st] = {**metrics(g, p, pr, v['loss'] / len(g)), 'pred': p, 'prob1': pr, 'gold': g}
    return out
def sel_key(m): return (m['balanced_accuracy'], m['macro_f1'], -m['loss'])
def train_run(regime, seed):
    seed_everything(seed); model = HoulsbyModel(regime).to(DEVICE)
    train_ds, train_ld = loader('train', seed, training=True); _, val_ld = loader('val', seed)
    head = model.adapter_params() + list(model.scorer.parameters())
    groups_ = [{'params': head, 'lr': LR_FROZEN if regime == 'frozen' else ADAPTER_SCORER_LR}]
    if regime == 'fullft': groups_.insert(0, {'params': model.encoder_core_params(), 'lr': ENCODER_LR})
    opt = torch.optim.AdamW(groups_, weight_decay=WEIGHT_DECAY); trainable = [p for g in groups_ for p in g['params']]
    upd = math.ceil(len(train_ld) / GRAD_ACCUM_STEPS) * EPOCHS; sch = get_linear_schedule_with_warmup(opt, int(upd * WARMUP_RATIO), upd)
    best = {'key': None}; hist = []; t0 = time.time(); torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch); model.train(); opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_ld, 1):
            ids = batch['input_ids'].to(DEVICE); mask = batch['attention_mask'].to(DEVICE); labels = batch['labels'].to(DEVICE); gold = batch['gold'].to(DEVICE); bsz, nopt, L = ids.shape
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(ids.view(bsz * nopt, L), mask.view(bsz * nopt, L), state='on').view(bsz, nopt)
                loss = (F.cross_entropy(logits.float(), labels, reduction='none') * CLASS_WEIGHT[gold]).mean()
            if not torch.isfinite(loss): raise FloatingPointError(f'non-finite loss: {regime} seed {seed} epoch {epoch} step {step}')
            (loss / GRAD_ACCUM_STEPS).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_ld):
                torch.nn.utils.clip_grad_norm_(trainable, 1.0); opt.step(); opt.zero_grad(set_to_none=True); sch.step()
            if step % 1000 == 0: print(f'  {TASK} {regime} seed {seed} epoch {epoch} step {step}/{len(train_ld)}')
        v = evaluate(model, val_ld, ['on'])['on']
        row = {'regime': regime, 'seed': seed, 'epoch': epoch, **{f'val_{k}': x for k, x in v.items() if k not in ('pred', 'prob1', 'gold')}}; hist.append(row)
        print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in row.items()})
        if best['key'] is None or sel_key(v) > best['key']:
            state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()} if regime == 'fullft' else {k: t.detach().cpu().clone() for k, t in model.state_dict().items() if ('.adapter.' in k or k.startswith('scorer'))}
            best = {'key': sel_key(v), 'state': state, 'epoch': epoch, 'val': {k: x for k, x in v.items() if k not in ('pred', 'prob1', 'gold')}}
    model.load_state_dict(best['state'], strict=(regime == 'fullft')); model.eval()
    return model, best, hist, (time.time() - t0) / 60, torch.cuda.max_memory_allocated() / 1024 ** 3

In [ ]:
# Runs (restart-safe per regime × seed). The evaluation set is scored once per selected checkpoint, adapters on and off.
def prog_path(regime, seed): return PROGRESS / f'{regime}_seed{seed}.json'
_, EVAL_LD = loader('eval', 0)
for seed in SEEDS:
    for regime in REGIMES:
        if prog_path(regime, seed).is_file(): print(f'{regime} seed {seed}: done, skipping'); continue
        print(f'\n===== {TASK.upper()} | HOULSBY | {regime} | seed {seed} =====')
        model, best, hist, minutes, peak = train_run(regime, seed)
        ev = evaluate(model, EVAL_LD, ['on', 'off'])
        st_file = STATES / f'{regime}_seed{seed}.pt'; saved = None
        try:
            keep = best['state'] if (regime == 'frozen' or SAVE_FULL_FT_STATES) else {k: v for k, v in best['state'].items() if ('.adapter.' in k or k.startswith('scorer'))}
            torch.save({'task': TASK, 'regime': regime, 'seed': seed, 'best_epoch': best['epoch'], 'state_dict': keep}, st_file)
            saved = {'file': str(st_file), 'bytes': st_file.stat().st_size, 'sha256': sha256_file(st_file), 'contents': 'full model' if (regime == 'fullft' and SAVE_FULL_FT_STATES) else 'adapters + scorer'}
        except Exception as exc:
            saved = {'error': repr(exc)}; print('WARNING: state not saved:', exc)
        rec = {'task': TASK, 'regime': regime, 'seed': seed, 'best_epoch': best['epoch'], 'best_val': best['val'], 'history': hist, 'minutes': minutes, 'peak_gpu_gb': peak, 'gpu': GPU_NAME, 'state': saved,
               'states': {st: {'metrics': {k: x for k, x in ev[st].items() if k not in ('pred', 'prob1', 'gold')}, 'pred': ev[st]['pred'].tolist(), 'prob1': ev[st]['prob1'].tolist()} for st in ('on', 'off')}}
        prog_path(regime, seed).write_text(json.dumps(rec, default=float))
        print(f"{regime} seed {seed}: on {100*ev['on']['balanced_accuracy']:.2f} | off {100*ev['off']['balanced_accuracy']:.2f} | epoch {best['epoch']} | {minutes:.1f} min")
        del model, best, ev; gc.collect(); torch.cuda.empty_cache()
print('All runs complete.')

In [ ]:
# Analysis per manifest H-1.
R = {(rg, s): json.loads(prog_path(rg, s).read_text()) for rg in REGIMES for s in SEEDS}
def BA(rg, s, st): return 100 * R[(rg, s)]['states'][st]['metrics']['balanced_accuracy']
def PRED(rg, s, st): return np.array(R[(rg, s)]['states'][st]['pred'])
# archived no-adapter full fine-tuning (same split, order and gold) — gate: exact gold/order match
if TASK == 'boolq':
    base = pd.read_csv(REPO_DIR / 'results' / 'fmi_nb2_fullft_results_20260925T081859Z' / 'fullft_predictions.csv'); base = base[(base.arm == 'none') & (base.state == 'on')]
else:
    base = pd.read_csv(REPO_DIR / 'results' / 'fmi_nb5_winogrande_results_20260930T222505Z' / 'dev_predictions.csv'); base = base[(base.method == 'fullft_none') & (base.state == 'on')]
NONE = {}
for s in sorted(base.seed.unique()):
    b = base[base.seed == s].sort_values('example_index'); b = b[b.example_index < len(GOLD)]
    if not np.array_equal(b.gold.to_numpy(), GOLD): raise RuntimeError(f'GATE: archived gold/order mismatch for seed {s}')
    NONE[int(s)] = b.prediction.to_numpy()
print('Gate passed: archived no-adapter predictions aligned for seeds', sorted(NONE))
def ba_pred(pred): return 100 * balanced_accuracy_score(GOLD, pred)
T = {2: 4.302652729911275, 4: 2.7764451051977987}
def ci(v):
    v = np.asarray(v, float); m = float(v.mean())
    if len(v) < 2: return {'mean': m, 'sd': float('nan'), 'ci95_low': float('nan'), 'ci95_high': float('nan'), 'values': v.tolist(), 'positive_seeds': int((v > 0).sum())}
    sd = float(v.std(ddof=1)); h = T.get(len(v) - 1, 4.302652729911275) * sd / math.sqrt(len(v)); return {'mean': m, 'sd': sd, 'ci95_low': m - h, 'ci95_high': m + h, 'values': [float(x) for x in v], 'positive_seeds': int((v > 0).sum())}
rows = []
for s in SEEDS:
    r_ = {'seed': s}
    for rg in REGIMES:
        r_[f'{rg}_on'] = BA(rg, s, 'on'); r_[f'{rg}_off'] = BA(rg, s, 'off'); r_[f'{rg}_on_minus_off'] = r_[f'{rg}_on'] - r_[f'{rg}_off']; r_[f'{rg}_epoch'] = R[(rg, s)]['best_epoch']
    r_['delta_regime'] = r_['frozen_on_minus_off'] - r_['fullft_on_minus_off']
    r_['no_adapter_ft'] = ba_pred(NONE[s]) if s in NONE else float('nan'); r_['D2_fullft_minus_no_adapter'] = r_['fullft_on'] - r_['no_adapter_ft'] if s in NONE else float('nan')
    rows.append(r_)
SEEDTAB = pd.DataFrame(rows); SEEDTAB.to_csv(RESULT_DIR / f'houlsby_{TASK}_by_seed.csv', index=False)
Q = {k: ci(SEEDTAB[k].dropna()) for k in ['frozen_on', 'frozen_off', 'frozen_on_minus_off', 'fullft_on', 'fullft_off', 'fullft_on_minus_off', 'delta_regime', 'no_adapter_ft', 'D2_fullft_minus_no_adapter']}
# gates
gate_frozen = Q['frozen_on']['mean'] > 100 * HEAD_ONLY_MEAN_BA
bv = {s: max(h['val_balanced_accuracy'] for h in R[('fullft', s)]['history']) for s in SEEDS}
gate_ft = all(v >= 0.60 for v in bv.values()) and all(BA('fullft', s, 'on') > 50 for s in SEEDS) and Q['fullft_on']['mean'] > 100 * HEAD_ONLY_MEAN_BA
rule = bool(gate_frozen and gate_ft and Q['delta_regime']['positive_seeds'] == len(SEEDS) and Q['delta_regime']['ci95_low'] > 0)
d2 = Q['D2_fullft_minus_no_adapter']; d2_word = 'no measurable change' if (math.isnan(d2['ci95_low']) or d2['ci95_low'] <= 0 <= d2['ci95_high']) else ('adapters add to full fine-tuning' if d2['ci95_low'] > 0 else 'adapters reduce full fine-tuning performance')
# shared cluster bootstrap
pos = GOLD == 1; cols, cidx = [], {}
def add(key, vec): cidx[key] = len(cols); cols.append(np.bincount(CLUSTER, weights=vec.astype(float), minlength=N_CLUSTERS))
add('np', pos.astype(float)); add('nn', (~pos).astype(float))
for s in SEEDS:
    for rg in REGIMES:
        for st in ('on', 'off'):
            c = PRED(rg, s, st) == GOLD; add((rg, s, st, 'p'), (c & pos).astype(float)); add((rg, s, st, 'n'), (c & ~pos).astype(float))
    if s in NONE:
        c = NONE[s] == GOLD; add(('none', s, 'on', 'p'), (c & pos).astype(float)); add(('none', s, 'on', 'n'), (c & ~pos).astype(float))
M = np.stack(cols, axis=1); rng = np.random.default_rng(BOOTSTRAP_SEED); acc = {'frozen_on_minus_off': [], 'fullft_on_minus_off': [], 'delta_regime': [], 'D2': []}
for _ in range(BOOTSTRAP_REPLICATES if not SMOKE_TEST else 200):
    S = M[rng.integers(0, N_CLUSTERS, N_CLUSTERS)].sum(axis=0); b = lambda key: 50 * (S[cidx[key + ('p',)]] / S[cidx['np']] + S[cidx[key + ('n',)]] / S[cidx['nn']])
    fz = [b(('frozen', s, 'on')) - b(('frozen', s, 'off')) for s in SEEDS]; ft = [b(('fullft', s, 'on')) - b(('fullft', s, 'off')) for s in SEEDS]
    acc['frozen_on_minus_off'].append(np.mean(fz)); acc['fullft_on_minus_off'].append(np.mean(ft)); acc['delta_regime'].append(np.mean(fz) - np.mean(ft))
    d2s = [b(('fullft', s, 'on')) - b(('none', s, 'on')) for s in SEEDS if s in NONE]
    if d2s: acc['D2'].append(np.mean(d2s))
BOOT = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in acc.items() if v}
def exact_mcnemar_p(x, y):
    n = x + y
    if n == 0: return 1.0
    k = min(x, y); return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
mcn = []
for s in SEEDS:
    for rg in REGIMES:
        a, b_ = PRED(rg, s, 'on') == GOLD, PRED(rg, s, 'off') == GOLD; x, y = int((a & ~b_).sum()), int((~a & b_).sum()); mcn.append({'regime': rg, 'seed': s, 'on_only': x, 'off_only': y, 'p_exact': exact_mcnemar_p(x, y)})
pd.DataFrame(mcn).to_csv(RESULT_DIR / f'houlsby_{TASK}_mcnemar.csv', index=False)
pred_rows = [{'task': TASK, 'regime': rg, 'seed': s, 'state': st, 'example_index': i, 'cluster': int(CLUSTER[i]), 'gold': int(GOLD[i]), 'prediction': int(p), 'prob_sem1': float(q)}
             for rg in REGIMES for s in SEEDS for st in ('on', 'off') for i, (p, q) in enumerate(zip(R[(rg, s)]['states'][st]['pred'], R[(rg, s)]['states'][st]['prob1']))]
pd.DataFrame(pred_rows).to_csv(RESULT_DIR / f'houlsby_{TASK}_predictions.csv', index=False)
pd.DataFrame([h for rg in REGIMES for s in SEEDS for h in R[(rg, s)]['history']]).to_csv(RESULT_DIR / f'houlsby_{TASK}_training_history.csv', index=False)
f = lambda q: f"{q['mean']:+.2f} [{q['ci95_low']:+.2f}, {q['ci95_high']:+.2f}]"
lines = [f'| Regime | on | off | on − off [seed CI] | cluster CI |', '|---|---:|---:|---|---|',
         f"| Frozen + Houlsby | {Q['frozen_on']['mean']:.2f} | {Q['frozen_off']['mean']:.2f} | {f(Q['frozen_on_minus_off'])} | {BOOT['frozen_on_minus_off'][0]:+.2f}, {BOOT['frozen_on_minus_off'][1]:+.2f} |",
         f"| Full FT + Houlsby | {Q['fullft_on']['mean']:.2f} | {Q['fullft_off']['mean']:.2f} | {f(Q['fullft_on_minus_off'])} | {BOOT['fullft_on_minus_off'][0]:+.2f}, {BOOT['fullft_on_minus_off'][1]:+.2f} |",
         f"| Frozen − full FT (regime) | | | {f(Q['delta_regime'])} | {BOOT['delta_regime'][0]:+.2f}, {BOOT['delta_regime'][1]:+.2f} |",
         f"| Full FT, no adapter (archived) | {Q['no_adapter_ft']['mean']:.2f} | | D2 {f(d2)} | {(str(round(BOOT['D2'][0],2)) + ', ' + str(round(BOOT['D2'][1],2))) if 'D2' in BOOT else '—'} |"]
TABLE = '\n'.join(lines); (RESULT_DIR / f'houlsby_{TASK}_table.md').write_text(TABLE + '\n')
summary = {'manifest': MANIFEST_ID, 'task': TASK, 'smoke_test': SMOKE_TEST, 'seeds': SEEDS, 'adapter_parameters': N_ADAPTER, 'seed_level': Q, 'cluster_bootstrap_95': BOOT,
           'gates': {'frozen_validity': bool(gate_frozen), 'fullft_optimization': bool(gate_ft), 'fullft_best_internal_val_BA': bv}, 'regime_rule_met': rule, 'D2_wording': d2_word,
           'fullft_trajectories_on_minus_off': SEEDTAB['fullft_on_minus_off'].round(3).tolist()}
(RESULT_DIR / f'houlsby_{TASK}_summary.json').write_text(json.dumps(summary, indent=2, default=float))
print(TABLE); print('\nRegime rule met:', rule, '| gates:', summary['gates']); print('D2:', d2_word); display(SEEDTAB.round(2))

In [ ]:
env = {'notebook': 'NB-7 H-1 Houlsby adapters', 'task': TASK, 'manifest': MANIFEST_ID, 'smoke_test': SMOKE_TEST, 'repository_commit': REPO_COMMIT, 'gpu': GPU_NAME, 'amp_dtype': str(AMP_DTYPE),
       'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': __import__('transformers').__version__, 'seeds': SEEDS, 'epochs': EPOCHS, 'bottleneck': BOTTLENECK,
       'run_started_utc': RUN_STARTED_UTC, 'run_finished_utc': datetime.now(timezone.utc).isoformat(), 'session_active_seconds': time.perf_counter() - RUN_STARTED_PERF,
       'runs': {f'{rg}_seed{s}': {k: R[(rg, s)].get(k) for k in ('minutes', 'peak_gpu_gb', 'best_epoch', 'state')} for rg in REGIMES for s in SEEDS}}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(env, indent=2, default=str))
for rg in REGIMES:
    for s in SEEDS: shutil.copy2(prog_path(rg, s), RESULT_DIR / f'run_{rg}_seed{s}.json')
rows_ = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'FILE_MANIFEST_SHA256.csv']
pd.DataFrame(rows_).to_csv(RESULT_DIR / 'FILE_MANIFEST_SHA256.csv', index=False)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); zip_path = Path(shutil.make_archive(f"/content/fmi_nb7_houlsby_{TASK}_{'smoke_' if SMOKE_TEST else ''}results_{stamp}", 'zip', RESULT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    if zf.testzip(): raise RuntimeError('ZIP corrupt')
shutil.copy2(zip_path, DRIVE_ROOT / zip_path.name); print('Result ZIP:', zip_path, '| sha256', sha256_file(zip_path))
from google.colab import files
files.download(str(zip_path))